In [1]:
import pandas as pd
import numpy as np
import ast

# Load data
movies = pd.read_csv("tmdb_5000_movies.csv")

keep_cols = [
    "title",
    "genres",
    "keywords",
    "original_language",
    "popularity",
    "vote_average",
    "vote_count",
    "budget",
    "revenue",
    "runtime"
]

movies = movies[keep_cols].copy()

# Clean data to drop NaNs
movies = movies.dropna(subset=["title", "genres", "keywords", "runtime", "vote_average", "vote_count"])
movies = movies.reset_index(drop=True)


# JSON parsing
def extract_names(text):
    try:
        items = ast.literal_eval(text)
        return [item["name"].strip().lower() for item in items if "name" in item]
    except:
        return []

# Apply cleaned keys to dataframe
movies["genre_list"] = movies["genres"].apply(extract_names)
movies["keyword_list"] = movies["keywords"].apply(extract_names)

# Keep movies with at least some parsed content
movies = movies[
    (movies["genre_list"].apply(len) > 0) | (movies["keyword_list"].apply(len) > 0)
].reset_index(drop=True)


# Vocab for content-based features
all_genres = sorted(set(g for row in movies["genre_list"] for g in row))

keyword_counts = {}
for row in movies["keyword_list"]:
    for kw in row:
        keyword_counts[kw] = keyword_counts.get(kw, 0) + 1

TOP_K_KEYWORDS = 300
top_keywords = sorted(keyword_counts, key=keyword_counts.get, reverse=True)[:TOP_K_KEYWORDS]

top_languages = movies["original_language"].value_counts().head(10).index.tolist()

genre_to_idx = {g: i for i, g in enumerate(all_genres)}
keyword_to_idx = {k: i + len(all_genres) for i, k in enumerate(top_keywords)}
language_to_idx = {
    lang: i + len(all_genres) + len(top_keywords)
    for i, lang in enumerate(top_languages)
}

n_content_features = len(all_genres) + len(top_keywords) + len(top_languages)


# Min max scaling
def min_max_scale(arr):
    arr = np.array(arr, dtype=float)
    min_val = np.min(arr)
    max_val = np.max(arr)
    if max_val == min_val:
        return np.zeros_like(arr)
    return (arr - min_val) / (max_val - min_val)


# Movie feature matrix
n_movies = len(movies)
X = np.zeros((n_movies, n_content_features + 3)) # Popularity, vote_average, runtime

for i in range(n_movies):
    for g in movies.loc[i, "genre_list"]:
        if g in genre_to_idx:
            X[i, genre_to_idx[g]] = 1.0

    for kw in movies.loc[i, "keyword_list"]:
        if kw in keyword_to_idx:
            X[i, keyword_to_idx[kw]] = 1.0

    lang = movies.loc[i, "original_language"]
    if lang in language_to_idx:
        X[i, language_to_idx[lang]] = 1.0

pop_scaled = min_max_scale(movies["popularity"].fillna(0).to_numpy(dtype=float))
vote_avg_scaled = min_max_scale(movies["vote_average"].fillna(0).to_numpy(dtype=float))
runtime_scaled = min_max_scale(
    movies["runtime"].fillna(movies["runtime"].median()).to_numpy(dtype=float)
)

X[:, -3] = pop_scaled
X[:, -2] = vote_avg_scaled
X[:, -1] = runtime_scaled

# Apply slight downweighting to numeric columns
X[:, -3:] *= 0.5

# Cosine similarity
def cosine_similarity_matrix(X):
    dot_products = X @ X.T
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    denom = norms @ norms.T

    sim = np.divide(
        dot_products,
        denom,
        out=np.zeros_like(dot_products),
        where=denom != 0
    )
    return sim

similarity_matrix = cosine_similarity_matrix(X)

# Success score using natural log of one plus the input 
def safe_log1p(arr):
    return np.log1p(np.maximum(arr, 0))

budget = movies["budget"].fillna(0).to_numpy(dtype=float)
revenue = movies["revenue"].fillna(0).to_numpy(dtype=float)
vote_count = movies["vote_count"].fillna(0).to_numpy(dtype=float)
vote_average = movies["vote_average"].fillna(0).to_numpy(dtype=float)

# ROI calculation, avoids division by zero 
roi = np.zeros_like(revenue, dtype=float)
mask = budget > 0
roi[mask] = revenue[mask] / budget[mask] # Revenue / budget is measure of success

roi_scaled = min_max_scale(safe_log1p(roi))
revenue_scaled = min_max_scale(safe_log1p(revenue))
vote_avg_scaled_success = min_max_scale(vote_average)
vote_count_scaled = min_max_scale(safe_log1p(vote_count))

success_score = (
    0.40 * roi_scaled +
    0.25 * revenue_scaled +
    0.20 * vote_avg_scaled_success +
    0.15 * vote_count_scaled
)

movies["success_score"] = success_score


# Title lookup helper
def find_movie_index(title, movies_df):
    exact = movies_df[movies_df["title"].str.lower() == title.lower()]
    if len(exact) > 0:
        return exact.index[0]

    partial = movies_df[movies_df["title"].str.lower().str.contains(title.lower(), na=False)]
    if len(partial) > 0:
        return partial.index[0]

    return None

# Content based recommender
def recommend_content_based(title, movies_df, sim_matrix, top_n=10):
    idx = find_movie_index(title, movies_df)

    if idx is None:
        print(f"Movie not found: {title}")
        return None

    scores = sim_matrix[idx].copy()
    scores[idx] = -1

    top_idx = np.argsort(scores)[::-1][:top_n]

    results = movies_df.loc[top_idx, ["title", "success_score"]].copy()
    results["similarity"] = scores[top_idx]

    results["similarity"] = results["similarity"].round(3)
    results["success_score"] = results["success_score"].round(3)

    return results[["title", "similarity", "success_score"]].reset_index(drop=True)

# Success aware recommender
def recommend_success_aware(
    title,
    movies_df,
    sim_matrix,
    alpha=0.75,
    beta=0.25,
    top_n=10
):
    idx = find_movie_index(title, movies_df)

    if idx is None:
        print(f"Movie not found: {title}")
        return None

    sim_scores = sim_matrix[idx].copy()
    sim_scores[idx] = -1

    final_scores = alpha * sim_scores + beta * movies_df["success_score"].to_numpy(dtype=float)
    final_scores[idx] = -1

    top_idx = np.argsort(final_scores)[::-1][:top_n]

    results = movies_df.loc[top_idx, ["title", "success_score"]].copy()
    results["similarity"] = sim_scores[top_idx]
    results["final_score"] = final_scores[top_idx]

    results["final_score"] = results["final_score"].round(3)
    results["similarity"] = results["similarity"].round(3)
    results["success_score"] = results["success_score"].round(3)

    return results[["title", "final_score", "similarity", "success_score"]].reset_index(drop=True)


# Comparison of the two methods
def compare_recommenders(title, movies_df, sim_matrix, top_n=5):
    print(f"\nINPUT MOVIE: {title}\n")

    print("CONTENT-BASED RECOMMENDATIONS")
    print(recommend_content_based(title, movies_df, sim_matrix, top_n=top_n))

    print("\nSUCCESS-AWARE RECOMMENDATIONS")
    print(recommend_success_aware(title, movies_df, sim_matrix, top_n=top_n))

# Examples
compare_recommenders("Avatar", movies, similarity_matrix, top_n=5)
print()
compare_recommenders("The Dark Knight", movies, similarity_matrix, top_n=5)
print()
compare_recommenders("Fight Club", movies, similarity_matrix, top_n=5)


INPUT MOVIE: Avatar

CONTENT-BASED RECOMMENDATIONS
                                         title  similarity  success_score
0                            Jupiter Ascending       0.691          0.465
1  Pirates of the Caribbean: On Stranger Tides       0.653          0.534
2                                     Predator       0.635          0.525
3                               Small Soldiers       0.635          0.449
4                          Clash of the Titans       0.626          0.481

SUCCESS-AWARE RECOMMENDATIONS
                                         title  final_score  similarity  \
0                            Jupiter Ascending        0.634       0.691   
1  Pirates of the Caribbean: On Stranger Tides        0.623       0.653   
2                                     Predator        0.607       0.635   
3                          Clash of the Titans        0.590       0.626   
4                               Small Soldiers        0.588       0.635   

   success_score  
0  